In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
from scipy.stats import levene
import matplotlib.pyplot as plt

from staeda.models_comparison import (
    calc_classification_metrics,
    calc_regression_metrics,
    make_mcs_plot_grid,
    make_scatterplot,
    make_ci_plot_grid,
    make_normality_diagnostic,
    make_boxplots_parametric,
    make_boxplots_nonparametric,
    make_sign_plots_nonparametric,
    make_curve_plots,
    rm_tukey_hsd,
)

In [ ]:
# Make plot directory if it doesn't exist
plot_dir = Path("./plots")
plot_dir.mkdir(parents=True, exist_ok=True)
print(f"Plots will be saved to: {plot_dir.resolve()}")

In [ ]:
from collections import defaultdict

def clean_method_name(method: str, data: str = None) -> str:
    """Cleans method names for better visualization."""
    final_method = ''
    if 'protac-stan' in method.lower():
        return 'PROTAC-STAN'
    if 'xgb' in method.lower():
        final_method = 'XGB'
    if 'mlp' in method.lower():
        final_method = 'MLP'

    final_method += '-QR' if '_qr' in method.lower() else ''
    final_method += '-MVE' if '_mve' in method.lower() else ''
    final_method += '-BIN' if '_bin' in method.lower() else ''
    
    data_features = []
    
    data_info = method if data is None else data
    
    if 'fp512r16' in data_info.lower():
        data_features.append('FP')
    if '_desc' in data_info.lower():
        data_features.append('Mol-Desc')
    if '_poi_ord' in data_info.lower():
        data_features.append('POI-Ord')
    if '_lig_ord' in data_info.lower():
        data_features.append('E3-Ord')
    if '_cell_ord' in data_info.lower():
        data_features.append('Cell-Ord')
    if '_assay_time' in data_info.lower():
        data_features.append('Time')
    if '_cell_desc' in data_info.lower():
        data_features.append('Cell-Desc')
    if '_poi_seq' in data_info.lower():
        data_features.append('POI-Vec')
    if '_lig_seq' in data_info.lower():
        data_features.append('E3-Vec')
    if '_poi_emb' in data_info.lower():
        data_features.append('POI-ESM-S')
    if '_lig_emb' in data_info.lower():
        data_features.append('E3-ESM-S')
    if '_poi_pca44_lig_pca7' in data_info.lower():
        data_features.append('POI/E3-ESM-S-PCA')

    # Remove extra spaces
    final_method += ' ' + ' '.join(sorted(data_features))
    return final_method

# Get all files in the predictions directory
predictions_dir = Path("./predictions/")
prediction_files = list(predictions_dir.glob("*.csv"))

protac_stan_preds_dir = Path("./protac_stac/predictions/")
protac_stan_files = list(protac_stan_preds_dir.glob("*.csv"))

# Load all CSV files in the predictions directory via a for loop
method_data2file = defaultdict(list)
clean_method2file = defaultdict(set)
raw_methods = set()
results = []
for file in sorted(prediction_files):
    # Extract model name from filename
    model_name = file.stem.split("=")[1].split("-")[0]
    data_name = file.stem.split("=")[2].split("-")[0]
    raw_methods.add((model_name + ' ' + data_name, clean_method_name(model_name, data_name)))
    method_data2file[model_name + ' ' + data_name].append(file.stem)
    clean_method2file[clean_method_name(model_name, data_name)].add(model_name + '-data=' + data_name)

for file in sorted(protac_stan_files):
    model_name = 'PROTAC-STAN'
    raw_methods.add((model_name, clean_method_name(model_name)))
    method_data2file[model_name].append(file.stem)
    clean_method2file[clean_method_name(model_name, data_name)].add(model_name + '-data=' + data_name)

for raw_method, method in sorted(raw_methods, key=lambda x: x[1]):
    print(f"{method:20} -> {raw_method}")
    
# Check that all the raw_method are unique
raw_method_names = [rm for rm, m in raw_methods]
assert len(raw_method_names) == len(set(raw_method_names)), "Raw method names are not unique!"

# Select the best methods to compare
methods = [
    'PROTAC-STAN',
    'MLP-BIN Cell-Ord E3-Ord Mol-Desc POI-Vec Time',
    'XGB-BIN Cell-Ord E3-Ord Mol-Desc POI-Vec Time',
]

for method in methods:
    if method not in [m for rm, m in raw_methods]:
        raise ValueError(f"Method {method} not found in raw methods!")

# Check that all selected methods have the same number of files
num_files = [len(method_data2file[rm]) for rm, m in raw_methods if m in methods]
if len(set(num_files)) != 1:
    raise ValueError("Not all selected methods have the same number of files!")

print()
for m in methods:
    files = clean_method2file[m]
    for f in files:
        print(f"{m:40} -> {f}")

In [ ]:
# Get all files in the predictions directory
predictions_dir = Path("./predictions/")
prediction_files = list(predictions_dir.glob("*.csv"))

protac_stan_preds_dir = Path("./protac_stac/predictions/")
protac_stan_files = list(protac_stan_preds_dir.glob("*.csv"))

# Load all CSV files in the predictions directory via a for loop
results = []
for file in prediction_files:
    # Extract model name from filename
    model_name = file.stem.split("=")[1].split("-")[0]
    data_name = file.stem.split("=")[2].split("-")[0]
    
    clean_method = clean_method_name(model_name, data_name)
    if clean_method not in methods:
        continue
    
    df = pd.read_csv(file)
    df['method'] = clean_method

    # Rename columns for consistency
    df = df.rename(columns={'group': 'split', 'value_type': 'task', 'confidence': 'prob'})
    # Rename task column values, from 'dmax' to 'Dmax' and 'dc50' to 'DC50'
    df['task'] = df['task'].str.replace('dmax', 'Dmax')
    df['task'] = df['task'].str.replace('dc50', 'DC50')
    df['task'] = df['task'].str.replace('binary_class', 'bin')
    df['task'] = df['task'].str.replace('heldout', 'bin')

    # For XGBoost, the 'pred' column refers to probabilities, so we need to
    # rename it and then threshold it at 0.5 to get binary predictions
    if 'bin' in df['task'].unique()[0]:
        if 'prob' not in df.columns:
            df['prob'] = df['pred'].copy()
            df['pred'] = (df['prob'] >= 0.5).astype(int)

    if 'PROTAC-STAN' in file.stem:
        df['set'] = 'test' if 'heldout' in file.stem else 'val'
    else:
        df['set'] = 'test' if 'test' in file.stem else 'val'
    results.append(df)

for file in protac_stan_files:
    model_name = 'PROTAC-STAN'
    clean_method = clean_method_name(model_name)
    if clean_method not in methods:
        continue
    
    df = pd.read_csv(file)
    df['method'] = clean_method

    # Rename columns for consistency
    df = df.rename(columns={'group': 'split', 'value_type': 'task', 'confidence': 'prob'})
    # Rename task column values, from 'dmax' to 'Dmax' and 'dc50' to 'DC50'
    df['task'] = df['task'].str.replace('dmax', 'Dmax')
    df['task'] = df['task'].str.replace('dc50', 'DC50')
    df['task'] = df['task'].str.replace('binary_class', 'bin')
    df['task'] = df['task'].str.replace('multitask', 'bin')
    df['task'] = df['task'].str.replace('heldout', 'bin')

    df['set'] = 'test' if 'heldout' in file.stem else 'val'
    results.append(df)

results_df = pd.concat(results, ignore_index=True)

# For get the maximum number of folds for any method
max_folds = results_df.groupby('method')['fold'].nunique().max()

# Build a mask to keep only (method, task) pairs with the required number of folds
mask = []
for (method, task), group in results_df.groupby(['method', 'task']):
    num_folds = group['fold'].nunique()
    required_folds = max_folds
    if num_folds < max_folds:
        print(f"WARNING: Method '{method}' for task {task} has only {num_folds} folds (expected {required_folds})")
        mask.extend(group.index.tolist())

# Remove only the problematic (method, task) pairs
results_df = results_df.drop(mask).reset_index(drop=True)

# Print all methods
print("\nMethods found in results:")
for method in sorted(results_df['method'].unique()):
    print(f"- {method}")

def dc50_to_pdc50(x):
    """Convert DC50 in nM to pDC50."""
    return -np.log10(x * 1e-9 + 1e-12)

# Convert all task 'DC50' to pDC50 values by taking -log10, they are in nM
for task_group, group in results_df.groupby('task'):
    if task_group == 'DC50':
        results_df.loc[group.index, 'target'] = group['target'].apply(dc50_to_pdc50)
        results_df.loc[group.index, 'pred'] = group['pred'].apply(dc50_to_pdc50)
        
        if 'pred_lower' in results_df:
            results_df.loc[group.index, 'pred_lower'] = group['pred_lower'].apply(dc50_to_pdc50)
            results_df.loc[group.index, 'pred_upper'] = group['pred_upper'].apply(dc50_to_pdc50)

In [ ]:
print(results_df['set'].unique())
print(results_df['task'].unique())
print(results_df['pred'].unique())
if 'prob' in results_df.columns:
    print(results_df['prob'].unique())

In [ ]:
PRECISION_THRESHOLD = 0.5

metrics = {}
for task in ['bin']:
    for dset in ['test']:
        subset = results_df[results_df['set'] == dset]
        subset = subset[subset['task'] == task]
        
        if subset.empty:
            print(f"WARNING: No data for task {task} on set {dset}, skipping...")
            continue

        print(f"Computing metrics for task {task} on set {dset}...")        
        if 'bin' in task:
            df_metrics = calc_classification_metrics(subset, "fold", "target", "prob", "pred", precision_threshold=PRECISION_THRESHOLD)
        else:
            df_metrics = calc_regression_metrics(subset, "fold", "target", "pred", classification_cutoffs[task])

        metrics[(task, dset)] = df_metrics.copy()
print("Precomputed metrics for all tasks and datasets.")

# Check if 'recall' contains any NaN values
for (task, dset), df_metrics in metrics.items():
    for c in df_metrics.columns:
        if df_metrics[c].isna().any():
            print(f"WARNING: '{c}' column contains NaN values for task {task} on set {dset}")

In [ ]:
is_parameteric = {}
for task in ['bin']:
    for dset in ['test']:
        # Retrieve precomputed metrics
        if (task, dset) not in metrics:
            print(f"WARNING: No metrics for task {task} on set {dset}, skipping...")
            continue
        df_metrics = metrics[(task, dset)].copy()

        metric_ls = df_metrics.columns[3:]
        
        variances_by_method = df_metrics.groupby('method')[metric_ls].var()
        max_fold_diff = variances_by_method.max() / variances_by_method.min()

        print("=" * 80)
        print(f"Analysis of {task} models on {dset} set:")
        print("=" * 80)

        is_parameteric[(task, dset)] = True
        
        for metric, var_fold_diff in max_fold_diff.items():
            # Perform Levene's test
            # NOTE: Levene's test should be done first, but here we combine it
            # with the the variance fold difference calculation to check both at
            # once.
            groups = df_metrics.groupby('method')[metric].apply(list)
            stat, pvalue = levene(*groups)
            
            if pvalue < 0.05:
                if var_fold_diff > 9:
                    print(f'• {metric.upper():>7}: Levene p-value: {pvalue} | Max fold variance ratio: {var_fold_diff:.4f} --> Variances are NOT equal: non-parametric tests are required')
                    is_parameteric[(task, dset)] &= False
                else:
                    print(f'• {metric.upper():>7}: Levene p-value: {pvalue} | Max fold variance ratio: {var_fold_diff:.4f}')
            else:
                print(f'• {metric.upper():>7}: Levene p-value: {pvalue} | Max fold variance ratio: {var_fold_diff:.4f}')

In [ ]:
for task in ['bin']:
    for dset in ['test']:
        # Retrieve precomputed metrics
        if (task, dset) not in metrics:
            print(f"WARNING: No metrics for task {task} on set {dset}, skipping...")
            continue
        df_metrics = metrics[(task, dset)].copy()

        # NOTE: The following are the metric columns and will be used later
        metric_ls = df_metrics.columns[3:]
    
        print("=" * 80)
        print(f"Analysis of {task} models on {dset} set:")
        print("=" * 80)
        make_normality_diagnostic(df_metrics, metric_ls)
        plt.savefig(plot_dir / f"normality_diagnostic_{task}_{dset}.pdf", bbox_inches='tight')
        plt.show()

In [ ]:
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from scipy.stats import f_oneway
import textwrap


def wrap_method_names(method_name, width=10):
    """ Wrap method names to a given width for better visualization."""
    return textwrap.fill(method_name, width)

def run_anova(df_in, col, group_var="method"):
    res_list = []
    for k,v in df_in.groupby(group_var):
        res_list.append(v[col].values)
    return f_oneway(*res_list)[1]


def make_simultaneous_ci_plot(df_in, metric_list, group_col="method", alpha=0.05, direction_dict=None, precision_threshold=0.5):
    """ Create simultaneous confidence interval plots for multiple metrics using Tukey HSD test results.

    Args:
        df_in (pd.DataFrame): Input dataframe containing the data.
        metric_list (list of str): List of metric column names to create confidence interval plots for.
        group_col (str): The column name indicating the groups. Default is "method".
    """
    tuckey_metrics = {}
    for i, metric in enumerate(metric_list):        
        # If any NaN values are present, skip plotting
        if df_in[metric].isna().any():
            print(f"WARNING: Metric {metric} contains NaN values, skipping...")
            continue

        tuckey_metric = pairwise_tukeyhsd(endog=df_in[metric],
                                          groups=df_in[group_col],
                                          alpha=alpha)
        
        print(f"Tukey HSD results for metric '{metric}':")
        print(tuckey_metric)
        tuckey_metrics[metric] = tuckey_metric
        
    metric2name = {
        'mae': 'MAE',
        'mse': 'MSE',
        'r2': 'R2',
        'rho': "Spearman's Rho",
        'roc_auc': 'ROC-AUC',
        'pr_auc': 'PR-AUC',
        'mcc': 'Matthews Correlation Coefficient (MCC)',
        'recall': f'Recall at Precision ≥ {precision_threshold}',
        'prec': 'Precision',
        'tnr': f'True Negative Rate (at Precision ≥ {precision_threshold})',
    }
        
    # Change the axes dimensions to be a bigger square
    fig, axes = plt.subplots(1, len(tuckey_metrics), figsize=(6 * len(tuckey_metrics), 5), sharey=True)
    
    for i, (metric, tuckey_metric) in enumerate(tuckey_metrics.items()):
        if direction_dict and metric in direction_dict:
            best_method = df_in.groupby(group_col)[metric].mean().reset_index().sort_values(by=metric, ascending=(direction_dict[metric]=='minimize')).iloc[0][group_col]
            metric_anova = run_anova(df_in, metric, group_var=group_col)
            if pd.isna(metric_anova):
                print(f"WARNING: ANOVA for metric {metric} returned NaN, skipping best method highlighting.")
                continue
            tuckey_metric.plot_simultaneous(comparison_name=best_method, ax=axes[i], figsize=(23, 2))
            axes[i].set_xlabel(metric2name.get(metric, metric), fontsize=12)
            axes[i].set_title(f"ANOVA p = {metric_anova:.2e}", fontsize=12)
            # Set x-axis ticks font size
            axes[i].tick_params(axis='x', labelsize=12)
            
            best_method = best_method.replace('\n', ' ')
            print(f"- Best method for {metric}: {best_method}")
        else:
            print(f"WARNING: No direction specified for metric {metric}, skipping best method highlighting.")
    
    # Change the font size of the y-axis labels
    for ax in axes:
        ax.tick_params(axis='y', labelsize=14)


direction_dict = {
    'roc_auc':'maximize',
    'pr_auc':'maximize',
    'mcc':'maximize',
    'recall':'maximize',
    'tnr':'maximize',
    'mae':'minimize',
    'mse':'minimize',
    'r2':'maximize',
    'rmse':'minimize',
    'rho':'maximize',
    'prec': 'maximize',
}

for task in ['bin']:
    for dset in ['test']:
        # Retrieve precomputed metrics
        if (task, dset) not in metrics:
            print(f"WARNING: No metrics for task {task} on set {dset}, skipping...")
            continue
        df_metrics = metrics[(task, dset)].copy()
        
        # df_metrics['method'] = df_metrics['method'].apply(lambda x: wrap_method_names(x, width=50))
        # Rename methods that start with 'MLP' or 'XGB' to shorter names for better visualization
        df_metrics['method'] = df_metrics['method'].replace({
            'MLP-BIN Cell-Ord E3-Ord Mol-Desc POI-Vec Time': 'MLP',
            'XGB-BIN Cell-Ord E3-Ord Mol-Desc POI-Vec Time': 'XGB',
        })
        metric_ls = df_metrics.columns[3:]
        
        print("=" * 80)
        print(f"Simultaneous Confidence Intervals of {task} models on {dset} set:")
        print("=" * 80)
        make_simultaneous_ci_plot(df_metrics, metric_ls, direction_dict=direction_dict, precision_threshold=PRECISION_THRESHOLD)
        plt.savefig(plot_dir / f"tukey_ci_{task}_{dset}.pdf", bbox_inches='tight')
        plt.show()

In [ ]:
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from scipy.stats import f_oneway
import textwrap
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np


def wrap_method_names(method_name, width=10):
    """ Wrap method names to a given width for better visualization."""
    return textwrap.fill(method_name, width)


def run_anova(df_in, col, group_var="method"):
    res_list = []
    for k, v in df_in.groupby(group_var):
        res_list.append(v[col].values)
    return f_oneway(*res_list)[1]


def set_size(width_pt, fraction=1, subplots=(1, 1)):
    """
    Set figure dimensions to avoid scaling in LaTeX.
    
    Parameters
    ----------
    width_pt: float
            Document width in points (result of \the\columnwidth)
    fraction: float, optional
            Fraction of the width which you wish the figure to occupy
    subplots: array-like, optional
            The number of rows and columns of subplots.
    
    Returns
    -------
    fig_dim: tuple
            Dimensions of figure in inches
    """
    # Width of figure (in pts)
    fig_width_pt = width_pt * fraction

    # Convert from pt to inches
    inches_per_pt = 1 / 72.27

    # Golden ratio to set aesthetic height
    golden_ratio = (5**.5 - 1) / 2

    # Figure width in inches
    fig_width_in = fig_width_pt * inches_per_pt
    
    # Figure height in inches
    fig_height_in = fig_width_in * golden_ratio * (subplots[0] / subplots[1])

    return (fig_width_in, fig_height_in)


def make_simultaneous_ci_plot(
    df_in,
    metric_list,
    group_col = "method",
    alpha = 0.05, 
    direction_dict = None,
    precision_threshold = 0.5,
    # LaTeX sizing parameters
    tex_width = 506.295,
    fraction = 1.0,
    use_constrained_layout = True,
    # Font parameters
    font_family = 'sans-serif',
    font_size = 9,
    title_font_size = None,
    xlabel_font_size = None,
    tick_font_size = None,
    title_fontweight = 'normal',
    xlabel_fontweight = 'bold',
    # Visual parameters
    grid_alpha = 0.3,
    subplot_label = 'b)',
    # Color parameters for error bars
    comparison_color = '#9DCE9C',  # blue for comparison group
    significant_color = '#C8ABDA',  # orange for significant differences
    nonsignificant_color = '#CCCCCC',  # gray for non-significant
    # Vertical reference line parameters
    vline_color = '#808080',  # gray for vertical reference lines
    vline_linewidth = 1.0,    # thickness of vertical lines
    vline_alpha = 0.5,        # transparency of vertical lines
):
    """ 
    Create simultaneous confidence interval plots for multiple metrics using Tukey HSD test results.
    Optimized for LaTeX/ACM publication format.

    Parameters:
    df_in (pd.DataFrame): Input dataframe containing the data.
    metric_list (list of str): List of metric column names to create confidence interval plots for.
    group_col (str): The column name indicating the groups. Default is "method".
    alpha (float): Significance level for the confidence intervals.
    direction_dict (dict): Dictionary mapping metric names to optimization direction ('maximize' or 'minimize').
    precision_threshold (float): Precision threshold for recall metric labeling.
    
    LaTeX sizing parameters:
    tex_width (float): LaTeX text/column width in points. Default 506.295pt (ACM text width).
    fraction (float): Fraction of tex_width to use for figure. Default 1.0.
    use_constrained_layout (bool): Use constrained layout (recommended for LaTeX). Default True.
    
    Font parameters:
    font_family (str): Font family for all text elements.
    font_size (int): Base font size (should match LaTeX body font, typically 9 for ACM).
    title_font_size (int): Font size for subplot titles. If None, uses font_size.
    xlabel_font_size (int): Font size for x-axis labels. If None, uses font_size.
    tick_font_size (int): Font size for axis tick labels. If None, uses font_size.
    title_fontweight (str): Font weight for subplot titles ('normal', 'bold', etc.).
    xlabel_fontweight (str): Font weight for x-axis labels ('normal', 'bold', etc.).
    
    Visual parameters:
    grid_alpha (float): Transparency for grid lines.
    subplot_label (str): Label for the first subplot (e.g., 'a)', 'b)').
    
    Returns:
    fig, axes
    """
    # Set default font sizes if not specified
    if title_font_size is None:
        title_font_size = font_size - 3
    if tick_font_size is None:
        tick_font_size = font_size - 2
    if xlabel_font_size is None:
        xlabel_font_size = font_size
    
    # Filter out 'tnr' from metric_list
    metric_list = [m for m in metric_list if m != 'tnr']
    
    tuckey_metrics = {}
    for i, metric in enumerate(metric_list):        
        # If any NaN values are present, skip plotting
        if df_in[metric].isna().any():
            print(f"WARNING: Metric {metric} contains NaN values, skipping...")
            continue

        tuckey_metric = pairwise_tukeyhsd(endog=df_in[metric],
                                          groups=df_in[group_col],
                                          alpha=alpha)
        
        tuckey_metrics[metric] = tuckey_metric
        
    metric2name = {
        'mae': 'MAE',
        'mse': 'MSE',
        'r2': 'R²',
        'rho': "Spearman's ρ",
        'roc_auc': 'ROC-AUC',
        'pr_auc': 'PR-AUC',
        'mcc': 'MCC',
        'recall': f'Recall (Prec. ≥ {precision_threshold})',
        'prec': 'Precision',
        'tnr': f'TNR (Prec. ≥ {precision_threshold})',
    }
    
    num_metrics = len(tuckey_metrics)
    if num_metrics == 0:
        print("WARNING: No valid metrics to plot.")
        return None, None
    
    # Set font globally
    plt.rcParams['font.family'] = font_family
    
    # Create figure with LaTeX-compatible sizing (2x2 grid)
    figsize = set_size(tex_width, fraction=fraction, subplots=(2, 2))
    w, h = figsize
    figsize = (w * 0.7, h * 0.8)  # Scale down to fit better in LaTeX column
    
    if use_constrained_layout:
        fig, axes = plt.subplots(2, 2, figsize=figsize, sharey=True, layout='constrained')
    else:
        fig, axes = plt.subplots(2, 2, figsize=figsize, sharey=True)
    axes = axes.flatten()
    
    for i, (metric, tuckey_metric) in enumerate(tuckey_metrics.items()):
        if i >= 4:  # Only plot first 4 metrics
            print(f"WARNING: More than 4 metrics provided. Only plotting first 4.")
            break
            
        if direction_dict and metric in direction_dict:
            best_method = df_in.groupby(group_col)[metric].mean().reset_index().sort_values(
                by=metric, ascending=(direction_dict[metric]=='minimize')
            ).iloc[0][group_col]
            
            metric_anova = run_anova(df_in, metric, group_var=group_col)
            if pd.isna(metric_anova):
                print(f"WARNING: ANOVA for metric {metric} returned NaN, skipping best method highlighting.")
                continue
                
            tuckey_metric.plot_simultaneous(
                comparison_name=best_method,
                ax=axes[i],
                figsize=figsize,
            )

            # Modify colors after plotting
            # Helper function to compare colors (handles both strings and arrays)
            def color_matches(color, target):
                if isinstance(color, str):
                    return color == target
                elif isinstance(color, np.ndarray):
                    # For arrays like [[0. 0. 1. 1.]], check if it matches matplotlib color
                    import matplotlib.colors as mcolors
                    target_rgba = mcolors.to_rgba(target)
                    return np.allclose(color.flatten(), target_rgba)
                return False
            
            # Update colors in containers (error bars)
            for container in axes[i].containers:
                if hasattr(container, 'get_children'):
                    for child in container.get_children():
                        current_color = child.get_color() if hasattr(child, 'get_color') else None
                        
                        if current_color is not None:
                            # Map original colors to new colors
                            if color_matches(current_color, 'b'):  # comparison group (blue)
                                child.set_color(comparison_color)
                            elif color_matches(current_color, 'r'):  # significant differences (red)
                                child.set_color(significant_color)
                            elif color_matches(current_color, '0.5'):  # non-significant (gray)
                                child.set_color(nonsignificant_color)
            
            # Also update line colors in the axes (includes error bars and vertical lines)
            for line in axes[i].get_lines():
                current_color = line.get_color()
                linestyle = line.get_linestyle()
                
                # Check if it's a vertical dashed line (reference line)
                if linestyle == '--' and color_matches(current_color, '0.7'):
                    line.set_color(vline_color)
                    line.set_linewidth(vline_linewidth)
                    line.set_alpha(vline_alpha)
                # Update error bar colors
                elif color_matches(current_color, 'b'):
                    line.set_color(comparison_color)
                elif color_matches(current_color, 'r'):
                    line.set_color(significant_color)
                elif color_matches(current_color, '0.5'):
                    line.set_color(nonsignificant_color)

            axes[i].set_xlabel(metric2name.get(metric, metric), 
                             fontsize=xlabel_font_size, fontweight=xlabel_fontweight)
            axes[i].set_title(f"p = {metric_anova:.2e}", 
                            fontsize=title_font_size, fontweight=title_fontweight)
            axes[i].tick_params(axis='x', labelsize=tick_font_size)
            axes[i].tick_params(axis='y', labelsize=tick_font_size)
            axes[i].grid(alpha=grid_alpha)
            
            best_method = best_method.replace('\n', ' ')
            best_value = df_in.groupby(group_col)[metric].mean().reset_index().sort_values(
                by=metric, ascending=(direction_dict[metric]=='minimize')
            ).iloc[0][metric]
            print(f"- Best method for {metric2name[metric]}: {best_method} - Value: {best_value:.4f}")
        else:
            print(f"WARNING: No direction specified for metric {metric}, skipping best method highlighting.")
    
    # Hide unused subplots if fewer than 4 metrics
    for j in range(i + 1, 4):
        axes[j].axis('off')

    # Add subplot label in the top-left corner
    if subplot_label:
        axes[0].text(-0.3, 0.95, subplot_label,
                     transform=axes[0].transAxes,
                     fontsize=font_size+2, fontweight='bold')
    
    return fig, axes


# Usage example
direction_dict = {
    'roc_auc':'maximize',
    'pr_auc':'maximize',
    'mcc':'maximize',
    'recall':'maximize',
    'tnr':'maximize',
    'mae':'minimize',
    'mse':'minimize',
    'r2':'maximize',
    'rmse':'minimize',
    'rho':'maximize',
    'prec': 'maximize',
}

for task in ['bin']:
    for dset in ['test']:
        # Retrieve precomputed metrics
        if (task, dset) not in metrics:
            print(f"WARNING: No metrics for task {task} on set {dset}, skipping...")
            continue
        df_metrics = metrics[(task, dset)].copy()
        
        # Rename methods that start with 'MLP' or 'XGB' to shorter names for better visualization
        df_metrics['method'] = df_metrics['method'].replace({
            'MLP-BIN Cell-Ord E3-Ord Mol-Desc POI-Vec Time': 'MLP',
            'XGB-BIN Cell-Ord E3-Ord Mol-Desc POI-Vec Time': 'XGB',
        })
        metric_ls = df_metrics.columns[3:]
        
        print("=" * 80)
        print(f"Simultaneous Confidence Intervals of {task} models on {dset} set:")
        print("=" * 80)
        fig, axes = make_simultaneous_ci_plot(
            df_metrics, 
            metric_ls, 
            direction_dict=direction_dict, 
            precision_threshold=PRECISION_THRESHOLD,
            # LaTeX sizing for ACM format
            tex_width=506.295,  # ACM text width
            # tex_width=241.14749,  # ACM column width
            fraction=1.0,       # Use full width
            font_size=10,        # ACM body font size
        )
        
        # Save WITHOUT bbox_inches='tight' to preserve exact dimensions
        plt.savefig(plot_dir / f"tukey_ci_{task}_{dset}.pdf")
        plt.savefig(plot_dir / f"tukey_ci_{task}_{dset}.svg")
        plt.show()

In [ ]:
for task in ['bin']:
    for dset in ['test']:
        # Retrieve precomputed metrics
        if (task, dset) not in metrics:
            print(f"WARNING: No metrics for task {task} on set {dset}, skipping...")
            continue
        df_metrics = metrics[(task, dset)].copy()
        
        # df_metrics['method'] = df_metrics['method'].apply(lambda x: wrap_method_names(x, width=20))
        # Rename methods that start with 'MLP' or 'XGB' to shorter names for better visualization
        df_metrics['method'] = df_metrics['method'].replace({
            'MLP-BIN Cell-Ord E3-Ord Mol-Desc POI-Vec Time': 'MLP',
            'XGB-BIN Cell-Ord E3-Ord Mol-Desc POI-Vec Time': 'XGB',
        })

        # NOTE: The following are the metric columns and will be used later
        metric_ls = df_metrics.columns[3:]

        if is_parameteric.get((task, dset)) is None:
            print(f"WARNING: No parametricity info for task {task} on set {dset}, skipping...")
            continue
        elif is_parameteric[(task, dset)]:
            print("=" * 80)
            print(f"Repeated measures ANOVA of {task} models on {dset} set:")
            print("=" * 80)
            make_boxplots_parametric(df_metrics.copy(), metric_ls, precision_threshold=PRECISION_THRESHOLD)
        else:
            print("=" * 80)
            print(f"Repeated non-parametric measures ANOVA of {task} models on {dset} set:")
            print("=" * 80)
            make_boxplots_nonparametric(df_metrics.copy(), metric_ls, precision_threshold=PRECISION_THRESHOLD)

        plt.savefig(plot_dir / f"boxplots_{task}_{dset}.pdf", bbox_inches='tight')
        plt.show()

In [ ]:
def text_sizes(num_methods):
    if num_methods <= 3:
        return {'cell_text_size': 16, 'axis_text_size': 16, 'title_text_size': 20}
    elif num_methods <= 4:
        return {'cell_text_size': 14, 'axis_text_size': 9, 'title_text_size': 18}
    elif num_methods <= 8:
        return {'cell_text_size': 9, 'axis_text_size': 8, 'title_text_size': 18}
    else:
        return {'cell_text_size': 8, 'axis_text_size': 7, 'title_text_size': 18}


# NOTE: For the effect sizes, provide the smallest difference that would be
# considered practically significant. 
effect_dicts = {
    'Dmax': {'mae': 15.0, 'mse': 15.0},
    'DC50': {'mae': 0.3, 'mse': 0.3}, # ~2-fold change in DC50, i.e., log10(2) ~= 0.3
    'bin': {'roc_auc': 0.1, 'pr_auc': 0.1, 'mcc': 0.1, 'recall': 0.1, 'tnr': 0.1},
    'Dmax_bin': {'roc_auc': 0.1, 'pr_auc': 0.1, 'mcc': 0.1, 'recall': 0.1, 'tnr': 0.1},
    'DC50_bin': {'roc_auc': 0.1, 'pr_auc': 0.1, 'mcc': 0.1, 'recall': 0.1, 'tnr': 0.1},
}

for task in ['bin']:
    for dset in ['test']:        
        # Retrieve precomputed metrics
        if (task, dset) not in metrics:
            print(f"WARNING: No metrics for task {task} on set {dset}, skipping...")
            continue
        df_metrics = metrics[(task, dset)].copy()
        
        # NOTE: The following are the metric columns and will be used later
        metric_ls = df_metrics.columns[3:]

        if is_parameteric[(task, dset)]:
            # # Wrap method names for better visualization
            # df_metrics['method'] = df_metrics['method'].apply(wrap_method_names)
            # Rename methods that start with 'MLP' or 'XGB' to shorter names for better visualization
            df_metrics['method'] = df_metrics['method'].replace({
                'MLP-BIN Cell-Ord E3-Ord Mol-Desc POI-Vec Time': 'MLP',
                'XGB-BIN Cell-Ord E3-Ord Mol-Desc POI-Vec Time': 'XGB',
            })

            print("=" * 80)
            print(f"Analysis of {task} models on {dset} set:")
            print("=" * 80)
            make_mcs_plot_grid(
                df_metrics.copy(),
                metric_ls,
                group_col="method",
                effect_dict=effect_dicts[task],
                figsize=(20, 12),
                # axis_text_size=16,
                # cell_text_size=16,
                # title_text_size=20,
                **text_sizes(len(df_metrics["method"].unique())),
                show_diff=True,
                sort_axes=True,
                direction_dict=direction_dict if 'bin' in task else {},
            )
        else:
            print("=" * 80)
            print(f"Non-parametric analysis of {task} models on {dset} set:")
            print("=" * 80)
            if 'bin' in task:
                # Issue a warning and skip for binary classification tasks
                print(f"WARNING: Skipping non-parametric sign plots for binary classification task {task}.")
                continue
            make_sign_plots_nonparametric(df_metrics.copy(), metric_ls)

        plt.savefig(plot_dir / f"effect_sizes_{task}_{dset}.pdf", bbox_inches='tight')
        plt.show()